# Week 11 — GradCAM and Deployment (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Amara: 'Before we deploy anything, I need to understand what the model is using as evidence.'

> 🧠 **What Will This Output?**
> GradCAM computes gradients w.r.t. the last conv feature map (4x4 spatial after four MaxPools), upsampled to 64x64. What detail can GradCAM NOT capture because of this resolution limit? How does occlusion sensitivity (12px patch, stride 4) compare?

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Maps for EfficientNet.
    Reference: Selvaraju et al. (2017) ICCV. arXiv:1610.02391
    """
    def __init__(self,model,target_layer):
        self.model=model; self.target_layer=target_layer
        self.gradients=None; self.activations=None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,"activations",o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"gradients",go[0].detach()))

    def generate(self,img_tensor,class_idx=None):
        self.model.eval()
        img_tensor=img_tensor.to(DEVICE)
        output=self.model(img_tensor)
        if class_idx is None: class_idx=output.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot=torch.zeros_like(output); one_hot[0,class_idx]=1.0
        output.backward(gradient=one_hot,retain_graph=True)
        weights=self.gradients.mean(dim=[2,3],keepdim=True)
        cam=torch.relu((weights*self.activations).sum(dim=1)).squeeze().cpu().numpy()
        if cam.ndim==0: cam=np.array([[float(cam)]])
        from PIL import Image as PILImg
        cam_img=PILImg.fromarray(cam).resize((img_tensor.shape[-1],img_tensor.shape[-2]),PILImg.BILINEAR)
        cam=np.array(cam_img)
        return (cam/cam.max() if cam.max()>0 else cam),class_idx

## Part 1 — GradCAM Visualisation

In [ ]:
gradcam=GradCAM(model,model.features[-1])

fig,axes=plt.subplots(4,6,figsize=(20,14))
for ri,land_use in enumerate(CLASSES):
    samples=df[(df["split"]=="test")&(df["land_use"]==land_use)].sample(2,random_state=42)
    for ci,(idx,row) in enumerate(samples.iterrows()):
        img_pil=Image.open(f"datasets/images/{row['filename']}").convert("RGB")
        img_t=ev(img_pil).unsqueeze(0)
        cam,pred_idx=gradcam.generate(img_t)
        pred_class=IDX2CLS[pred_idx]; true_class=row["land_use"]
        correct=pred_class==true_class; img_arr=np.array(img_pil)
        ax_orig=axes[ri][ci*3]; ax_cam=axes[ri][ci*3+1]; ax_ov=axes[ri][ci*3+2]
        ax_orig.imshow(img_arr); ax_orig.set_title(f"{row['city']}\nTrue:{true_class[:4]}",fontsize=7); ax_orig.axis("off")
        ax_cam.imshow(cam,cmap="jet",vmin=0,vmax=1); ax_cam.set_title("GradCAM",fontsize=7); ax_cam.axis("off")
        ax_ov.imshow(img_arr); ax_ov.imshow(cam,cmap="jet",alpha=0.5); 
        ax_ov.set_title(f"Pred:{pred_class[:4]} {'ok' if correct else 'X'}",fontsize=7,
                        color="green" if correct else "red"); ax_ov.axis("off")
plt.suptitle("GradCAM — Where the Model Looks (jet: red=high attention)",fontweight="bold",y=1.01)
plt.tight_layout(); plt.savefig("outputs/gradcam_grid.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 2 — Occlusion Sensitivity

> ⏸️ **Recall Prompt**
> Book 2 Week 11 documented a 26-point Pidgin gap in the model card. Book 3 documents a Kano equity gap. In what sense are these the same disclosure? What is the single most important sentence a model card must contain?

In [ ]:
def occlusion_sensitivity(model,img_pil,class_idx,patch_size=12,stride=4,device="cpu"):
    """Occlusion sensitivity: slide a grey patch, measure confidence drop.
    Reference: Zeiler & Fergus (2014) ECCV. arXiv:1311.2901
    """
    t=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    H,W=img_pil.size[1],img_pil.size[0]
    img_t=t(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        baseline_conf=torch.softmax(model(img_t),dim=1)[0,class_idx].item()
    sens_map=np.zeros((H,W)); count_map=np.zeros((H,W))
    for y in range(0,H-patch_size+1,stride):
        for x in range(0,W-patch_size+1,stride):
            occ=img_t.clone(); occ[0,:,y:y+patch_size,x:x+patch_size]=0.5
            with torch.no_grad():
                c=torch.softmax(model(occ),dim=1)[0,class_idx].item()
            drop=baseline_conf-c
            sens_map[y:y+patch_size,x:x+patch_size]+=drop
            count_map[y:y+patch_size,x:x+patch_size]+=1
    sens_map/=np.maximum(count_map,1)
    return sens_map,baseline_conf

In [ ]:
# Compare GradCAM vs Occlusion sensitivity on 4 patches (2 correct, 2 wrong)
test_df=df[df["split"]=="test"].reset_index(drop=True)
model.eval()  # ensure eval mode: Dropout off, BatchNorm uses running stats
all_preds,all_labels=[],[]
with torch.no_grad():
    loader=DataLoader(SatelliteDataset(test_df,"datasets/images",ev),64,False,num_workers=0)
    for imgs,lbls,_ in loader:
        all_preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy()); all_labels.extend(lbls.numpy())
test_df["pred"]=all_preds; test_df["true"]=all_labels
test_df["correct"]=test_df["pred"]==test_df["true"]

# Pick 2 correct + 2 wrong
correct_s=test_df[test_df["correct"]].sample(2,random_state=42)
wrong_s  =test_df[~test_df["correct"]].sample(min(2,sum(~test_df["correct"])),random_state=42)
sample4  =pd.concat([correct_s,wrong_s])

fig,axes=plt.subplots(len(sample4),3,figsize=(12,4*len(sample4)))
if len(sample4)==1: axes=[axes]
for ri,(_,row) in enumerate(sample4.iterrows()):
    img_pil=Image.open(f"datasets/images/{row['filename']}").convert("RGB")
    img_t=ev(img_pil).unsqueeze(0)
    cam,_=gradcam.generate(img_t,class_idx=int(row["true"]))
    sens,base_conf=occlusion_sensitivity(model,img_pil,int(row["true"]),device=DEVICE)
    img_arr=np.array(img_pil)
    axes[ri][0].imshow(img_arr)
    axes[ri][0].set_title(f"{'ok' if row['correct'] else 'X'} True:{IDX2CLS[row['true']]}",fontsize=9,
                           color="green" if row["correct"] else "red"); axes[ri][0].axis("off")
    axes[ri][1].imshow(img_arr); axes[ri][1].imshow(cam,cmap="jet",alpha=0.5)
    axes[ri][1].set_title("GradCAM (gradient attention)",fontsize=9); axes[ri][1].axis("off")
    sens_n=sens/sens.max() if sens.max()>0 else sens
    axes[ri][2].imshow(img_arr); axes[ri][2].imshow(sens_n,cmap="hot",alpha=0.5)
    axes[ri][2].set_title("Occlusion (necessary regions)",fontsize=9); axes[ri][2].axis("off")
plt.suptitle("GradCAM vs Occlusion Sensitivity — Attention vs Necessity",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/xai_comparison.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 3 — FastAPI Deployment

In [ ]:
APP_CODE = """
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import torch, torch.nn as nn, io
from torchvision import models, transforms
from PIL import Image

app = FastAPI(title="VisionAI Lagos Classifier", version="1.0")
CLASSES = ["commercial","green_space","industrial","residential"]
MEAN    = [0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
THRESHOLD = 0.70

def build_model(nc=4):
    m=models.efficientnet_b0(weights=None)
    m.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(1280,nc)); return m

model=build_model()
model.load_state_dict(torch.load("models/efficientnet_best.pth",map_location="cpu"))
model.eval()
tfm=transforms.Compose([transforms.Resize((64,64)),transforms.ToTensor(),
                          transforms.Normalize(MEAN,STD)])

@app.post("/classify")
async def classify(file: UploadFile=File(...)):
    try:
        img=Image.open(io.BytesIO(await file.read())).convert("RGB")
        t=tfm(img).unsqueeze(0)
        with torch.no_grad(): probs=torch.softmax(model(t),dim=1)[0]
        pred_idx=probs.argmax().item(); conf=probs[pred_idx].item()
        return {"land_use":CLASSES[pred_idx],"confidence":round(conf,3),
                "requires_review":conf<THRESHOLD,
                "all_probabilities":{c:round(p.item(),3) for c,p in zip(CLASSES,probs)}}
    except Exception as e: return JSONResponse(status_code=500,content={"error":str(e)})

@app.get("/health")
def health(): return {"status":"healthy","model":"EfficientNet-B0","version":"1.0"}
"""
with open("vision_api/app.py","w") as f: f.write(APP_CODE)
print("FastAPI app saved: vision_api/app.py")
print("Run: uvicorn vision_api.app:app --reload --port 8000")
print("Test: curl -X POST http://localhost:8000/classify -F 'file=@datasets/images/kano_industrial_00001.png'")

## Part 4 — Model Card

In [ ]:
MODEL_CARD="""
# Model Card — VisionAI Lagos Land-Use Classifier v1.0

## Model Details
Architecture: EfficientNet-B0 (ImageNet pretrained, fine-tuned)
Task: 4-class satellite land-use classification
Classes: commercial, green_space, industrial, residential
Input: 64x64 RGB satellite patches
Cities: Lagos, Ibadan, Kano, Abuja

## Performance
| Metric | Value |
|--------|-------|
| Overall weighted F1 | [FILL IN] |
| Lagos F1 | [FILL IN] |
| Ibadan F1 | [FILL IN] |
| Kano F1 | [FILL IN] |
| Abuja F1 | [FILL IN] |
| Equity gap | [FILL IN] |

## Fairness Findings
A [N]-point F1 gap exists between best and worst city.
Root cause: Kano arid terrain is spectrally distinct from coastal majority.
Mitigation: confidence threshold routing (patches < 0.70 -> human review).
Retraining commitment: 500 additional Kano patches within 60 days.

## Intended Use
Automated land-use classification for urban planning and change detection.

## NOT Recommended For
Legal land title decisions (human review required).
Applications where Kano performance gap is unacceptable.

## GradCAM Validation
[FILL IN after running Part 1: does model attention correspond to meaningful regions?]

## Monitoring
Weekly: overall F1 on held-out validation set.
Monthly: per-city F1 and equity gap.
Retrain trigger: Kano F1 < 0.65 OR overall F1 drop > 0.05.

Reference: Mitchell et al. (2019) Model Cards for Model Reporting. FaCCT 2019.
"""
with open("outputs/model_card.md","w") as f: f.write(MODEL_CARD)
print("Model card saved: outputs/model_card.md")
print(MODEL_CARD)

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| GradCAM with model.train() | model.eval() before generating |
| Not detaching in hooks | Memory leak |
| Comparing GradCAM across images | Normalise each map to [0,1] independently |
| Deploying without model.eval() at startup | Set once at server startup |

## Week 11 Complete

Next: `week12_capstone_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 11, you can now:

- Implement GradCAM and produce activation maps for correct and misclassified examples
- Explain in plain language what region the model uses for classification
- Apply occlusion sensitivity to find necessary image patches
- Write an explainability section suitable for a stakeholder report

📤 **GitHub:** Push week11_gradcam_STARTER.ipynb. Commit: "Week 11: GradCAM implemented, attention maps produced"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
